### トンプソンサンプリングとは
- ベイズ推定を用いて多腕バンディット問題において探索と活用のバランスを取る手法。
- 各腕の期待報酬の確率分布からサンプリングして腕を選ぶという確率論的なアプローチを取る。
- 不確実な腕ほど分布の広がりが大きく、高い値がサンプリングされる可能性が高いため、自然と探索が促される。

||内容|
|----|----|
|問題設定|多腕バンディット問題|
|事前分布|ベータ分布（α=β=1 で一様分布から開始）|
|更新|ベータ-ベルヌーイ共役事前分布（報酬に応じて αかβ を更新）|
|腕の選択|各腕の事後分布からサンプリングして最大値の腕を選ぶ|




In [ ]:
# numpyを用いたシミュレーション

import numpy as np

rng = np.random.default_rng(42)

n_arms   = 5
n_rounds = 1000
true_means = rng.uniform(0, 1, n_arms)

alphas = np.ones(n_arms)
betas  = np.ones(n_arms)
total_reward = 0

for _ in range(n_rounds):
    samples = rng.beta(alphas, betas)
    arm = np.argmax(samples)

    reward = rng.binomial(1, true_means[arm])
    alphas[arm] += reward
    betas[arm]  += 1 - reward
    total_reward += reward

print(f"真の期待報酬（各腕）: {true_means.round(4)}")
print(f"最適な腕のインデックス: {np.argmax(true_means)}")
print(f"総報酬: {total_reward}")

真の期待報酬（各腕）: [0.774  0.4389 0.8586 0.6974 0.0942]
最適な腕のインデックス: 2
総報酬: 832


In [ ]:
# scratch実装

import numpy as np


class ThompsonSampling:
    def __init__(self, n_arms, n_rounds, random_state=None):
        self.n_arms   = n_arms
        self.n_rounds = n_rounds
        self.rng      = np.random.default_rng(random_state)

        self.alphas  = np.ones(n_arms)
        self.betas   = np.ones(n_arms)
        self.history = []

    def select_arm(self):
        samples = self.rng.beta(self.alphas, self.betas)
        return np.argmax(samples)

    def update(self, arm, reward):
        self.alphas[arm] += reward
        self.betas[arm]  += 1 - reward

    def run(self, true_means):
        total_reward = 0
        for _ in range(self.n_rounds):
            arm    = self.select_arm()
            reward = self.rng.binomial(1, true_means[arm])
            self.update(arm, reward)
            total_reward += reward
            self.history.append(arm)
        return total_reward


# 実行
rng = np.random.default_rng(42)
n_arms     = 5
n_rounds   = 1000
true_means = rng.uniform(0, 1, n_arms)

bandit = ThompsonSampling(n_arms=n_arms, n_rounds=n_rounds, random_state=42)
total_reward = bandit.run(true_means)

print(f"真の期待報酬（各腕）: {true_means.round(4)}")
print(f"最適な腕のインデックス: {np.argmax(true_means)}")
print(f"各腕の選択回数: {bandit.alphas.astype(int) + bandit.betas.astype(int) - 2}")
print(f"総報酬: {total_reward}")


真の期待報酬（各腕）: [0.774  0.4389 0.8586 0.6974 0.0942]
最適な腕のインデックス: 2
各腕の選択回数: [103   8 877   8   4]
総報酬: 831


`select_arm` メソッド

各腕のベータ分布からサンプルを引いて最大値の腕を選びます。UCBのように数式でスコアを計算するのではなく、確率的なサンプリングで腕を選ぶのがトンプソンサンプリングの特徴です。

`update` メソッド

報酬が1なら α を、0なら β を1増やします。ベータ-ベルヌーイ共役事前分布の更新式そのままです。

選択回数の計算

$ α_a+β_a−2 $で選択回数が求まります。初期値が $ α=β=1 $ なので、試行回数は $(α−1)+(β−1)$ です。